In [0]:
use catalog kishoredb;
use schema cap_db;

### Task 1 
Incremental Data Loading using copy into command without manual schema



In [0]:
-- creating the table without using schema
create table orders_bronze;

In [0]:
-- Batch streaming data ingestion using the copy into cmd
copy into orders_bronze
from '/Volumes/kishoredb/cap_db/cap_files/incremental_data_files/'
fileformat =json
format_options('header'='true','inferSchema'='true')
copy_options('mergeSchema'='true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5019,5019,0


In [0]:
-- After adding Jan and feb dataset ingestion using copy into cmd
select count(*) as feb_cnt from orders_bronze ;

feb_cnt
11632


In [0]:
-- After adding mar dataset ingestion using copy into 
select count(*) as mar_cnt from orders_bronze;

mar_cnt
16651


### Task -2
Order Data analysis

In [0]:
-- 1.total no of orders placed 
select count(*) as total_orders from orders_bronze;

total_orders
16651


In [0]:
-- 2.orders by payment method
select payment_method,count(*) as total_orders from orders_bronze group by 1 

payment_method,total_orders
card,4186
UPI,4124
wallet,4140
cash,4201


In [0]:
-- 3.order status distribution
select order_status,count(*) as total_orders from orders_bronze group by order_status

order_status,total_orders
cancelled,4129
en route,4238
delivered,4105
preparing,4179


In [0]:
-- 4.orders per restaurant 
select restaurant_id,count(*) as total_orders from orders_bronze group by restaurant_id 

restaurant_id,total_orders
R060,81
R040,92
R123,100
R098,81
R070,86
R111,76
R136,86
R159,84
R131,108
R089,101


In [0]:
-- 5.orders per customer
select customer_id , count(*) as total_orders from orders_bronze group by customer_id

customer_id,total_orders
C5784,1
C5037,1
C2559,2
C0977,2
C0752,1
C1136,2
C1524,4
C9889,3
C7872,3
C4710,3


In [0]:
-- 6. orders handled per delivery agents
select agent_id,count(*) as total_orders from orders_bronze group by agent_id

agent_id,total_orders
A099,41
A165,34
A352,29
A050,29
A220,40
A360,79
A285,45
A143,28
A124,34
A400,44


In [0]:
-- 7.Top 10 orders by total amount
select order_id,round(sum(total_amount),2) as total_amt from orders_bronze group by 1 order by 2 desc limit 10

order_id,total_amt
c12860c3-683a-47f7-b790-7cbbbcfc0fec,361.46
05e56a7c-190c-4bde-ac13-e34b38fe155a,355.9
341cb220-46e4-4f0d-b2f2-2e494b5729dd,352.45
3b6f5bd2-08e4-412d-be9e-7701463015b0,348.95
f43228bd-94e5-4ef6-914a-c7ebdceaa572,342.91
621d746e-bd7d-4e27-b06e-a098a9bffda7,340.65
75dfeba8-60e0-42c2-9149-315f030e8d9d,333.85
72559328-7d0d-4177-8c44-4ebc43dca731,332.98
b9dc92fc-1922-4274-a42c-85b83e46bea2,327.8
7d9b6852-ffb2-4bac-84e4-cc7960284e30,327.44


In [0]:
-- 8. avg tip per payment method
select payment_method,round(avg(tip),2) as avg_tip  from orders_bronze group by payment_method 

payment_method,avg_tip
card,2.51
UPI,2.53
wallet,2.5
cash,2.47


In [0]:
-- 9.percent of orders  tip >10% of total_amount
select round(avg(case when tip>0.1*total_amount then 1.0 else 0 end) * 100.0 , 2) as percent_orders from orders_bronze;

percent_orders
21.49


In [0]:
-- 10.percentage of success vs cancelled vs failed orders
select round(avg(case when order_status='delivered' then 1.0 else 0 end)*100.0,2) as success_orders_percentage,
       round(sum(case when order_status='cancelled' then 1 else 0 end)*100.0 / count(*),2) as cancelled_orders_percentage,
       round(count(case when order_status='en route' then 1 end)*100.0 / count(*),2) as failed_orders_percentage       
from orders_bronze

success_orders_percentage,cancelled_orders_percentage,failed_orders_percentage
24.65,24.80,25.45


In [0]:
-- 11.top cust (no of orders) and top res (no of orders)
with cte1 as(select customer_id,count(*) as cust_total_orders from orders_bronze group by customer_id order by cust_total_orders desc limit 1),
cte2 as(
select restaurant_id,count(*) as res_total_orders from orders_bronze group by restaurant_id order by 2 desc limit 1)
select * from cte1,cte2;

customer_id,cust_total_orders,restaurant_id,res_total_orders
C7381,9,R005,114


In [0]:
-- creating a temp view bcoz reusage of this query again and again
create or replace temp view orders_exploded as 
select agent_id,customer_id,delivery_location_id,explode(items_ordered.item_id) as item_id ,explode(items_ordered.item_name) as item_name,explode(items_ordered.price) as price,explode(items_ordered.quantity) as quantity,order_id,order_status,payment_method,restaurant_id,tip,total_amount
from kishoredb.cap_db.orders_bronze

In [0]:
-- 12.Explode items_ordered
select * from orders_exploded  ;

agent_id,customer_id,delivery_location_id,item_id,item_name,price,quantity,order_id,order_status,payment_method,restaurant_id,tip,total_amount
A052,C7420,L049,I0117,Over Juice,0.0,5,47b006aa-585e-4407-a621-2da8e9a4da59,delivered,cash,R078,4.87,0.0
A270,C9458,L020,I0329,Job Ice Cream,6.66,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Job Ice Cream,6.66,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Job Ice Cream,24.58,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Job Ice Cream,24.58,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Follow Tea,6.66,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Follow Tea,6.66,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Follow Tea,24.58,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0329,Follow Tea,24.58,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72
A270,C9458,L020,I0251,Job Ice Cream,6.66,3,b293cb14-bdc2-4f3c-8ef3-4ee49cfe89a5,cancelled,cash,R144,3.19,93.72


In [0]:
-- 13. Most frequntly ordered items
select item_name,count(*) as cnt from orders_exploded group by item_name order by cnt desc limit 1

item_name,cnt
Magazine Pasta,1689


In [0]:
-- 14. Avg no of items per order
select round(avg(item_count),2) as avg_per_order from(select order_id, count(item_id) as item_count from orders_exploded group by order_id);

avg_per_order
18.39


In [0]:
-- 15.Find item revenue contribution per restaurant
select restaurant_id,item_name,round(sum(price*quantity),2) as item_revenue from orders_exploded group by restaurant_id,item_name 

restaurant_id,item_name,item_revenue
R070,Degree Ice Cream,34818.96
R001,Theory Spring Roll,26501.19
R074,Several Pizza,8328.01
R061,Born Biryani,1308.32
R146,She Tea,57393.51
R077,Red Tea,10750.25
R102,Market Spring Roll,7978.38
R112,Environment Nachos,37472.6
R167,Population Ice Cream,2522.8
R083,Spend Pizza,25774.95


### Task 3
order Data and Dimensions dataset analysis

In [0]:

-- 1.Find the preferred restaurant names for each customer
with cte as (select customer_id,restaurant_id,count(*) from orders_bronze group by 1,2 order by 3 desc )
select c.customer_id,r.name from cte c join restaurants_filtered r on c.restaurant_id=r.restaurant_id

customer_id,name
C0308,Pasta Palace Hub
C6488,Curry Leaf Grill
C8962,Pasta Palace Hub
C4442,Tasty Bites Express
C3246,Pasta Palace Kitchen
C0307,Curry Leaf Kitchen
C1797,Curry Leaf Corner
C3071,Pasta Palace Hub
C5302,Pasta Palace Grill
C1004,Curry Leaf Kitchen


In [0]:
-- 2.Find the top 10 areas receiving orders in each month
with cte as (select extract(month from o.timestamp) as month,l.area,count(*) as order_cnt from orders_bronze o join locations_filtered l on o.delivery_location_id=l.location_id group by extract(month from o.timestamp), l.area )
,
ranked as (
    select month,area,order_cnt,
     rank() over(partition by month order by order_cnt desc) as rnk
     from cte
)
select month,area,order_cnt from ranked where rnk<=10
order by month,order_cnt desc

month,area,order_cnt
1,Rodriguez Mill,87
1,Eric Hills,86
1,Hill Station,80
1,Smith Estates,79
1,Harper Street,76
1,Copeland Parks,75
1,Morgan Key,75
1,John Underpass,74
1,Jennifer Wall,73
1,Ashley Extensions,73


In [0]:
-- 3.Find all successfully delivered orders that were paid using a card to analyze payment preferences and delivery success rate

select order_id from orders_bronze where order_status='delivered' and payment_method='card'

order_id
e293e41d-e335-4d6b-8504-34f96140e660
c8bd7ee7-a451-447a-811b-fd69aff36062
e71e1b46-c026-4913-9b58-870fde58e72b
ede3312f-d227-42a5-a1d0-d191999abbd1
7e5fb9b0-c7e0-4080-89f3-6707df2a16a4
22accf1f-f43d-4051-9240-9a971c27bf85
bee913bc-f4fd-44b7-8f51-229c93ef4c5d
5d838bcb-e367-4734-93fe-c9a2551e440f
938839a3-20ac-4b3b-a9b1-fb9b364bc210
3cd888ae-1614-4d75-bd5c-037edd3684f4


In [0]:
-- creating a temp table for the cuisines bcoz restaurants having different type of cuisines list
create or replace temporary table restaurants_cuisines_filtered
as 
select restaurant_id,name,explode(cuisine) as cuisines,location_id,rating,delivery_fee 
from (select restaurant_id, name,split(cuisines, ',') AS cuisine,location_id,rating , delivery_fee
from restaurants_filtered);
select * from restaurants_cuisines_filtered ;

restaurant_id,name,cuisines,location_id,rating,delivery_fee
R001,Food Haven Grill,Indian,L069,3.3,3.9
R002,Spice Villa Hub,Thai,L012,3.8,2.25
R002,Spice Villa Hub,Italian,L012,3.8,2.25
R002,Spice Villa Hub,Chinese,L012,3.8,2.25
R004,Curry Leaf Hub,Indian,L048,3.1,3.03
R004,Curry Leaf Hub,American,L048,3.1,3.03
R005,Tasty Bites Grill,Mexican,L039,3.9,1.02
R007,Pasta Palace Corner,American,L014,3.3,3.33
R007,Pasta Palace Corner,Mexican,L014,3.3,3.33
R007,Pasta Palace Corner,Chinese,L014,3.3,3.33


In [0]:
-- 4.Which cuisines generate the highest avg order value 
select r.cuisines,round(avg(o.total_amount),2) as avg_order from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id group by r.cuisines order by avg_order desc limit 1

cuisines,avg_order
Indian,82.28


In [0]:
-- 4.Which cuisines generate the highest avg order value
-- select r.cuisines,round(avg(o.total_amount),2) as avg_order from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id group by r.cuisines order by avg_order desc limit 1

In [0]:
-- 5.Which delivery agents have delivered the most high-value order (above 50)
select a.agent_id,a.name,o.total_amount from orders_bronze o join delivery_agents_filtered a on o.agent_id=a.agent_id where o.order_status='delivered' and 
 o.total_amount>50  order by o.total_amount desc 

agent_id,name,total_amount
A339,Jason Hanson,355.9
A273,Emily Gutierrez,340.65
A273,Emily Gutierrez,340.65
A248,Mrs. Vanessa Gutierrez,327.8
A248,Mrs. Vanessa Gutierrez,327.8
A237,Ryan Patterson,320.03
A244,Cynthia Edwards,309.09
A201,Robert Webster,306.6
A393,Justin Lam,299.64
A393,Justin Lam,299.64


In [0]:
-- 6.Which cities (area_name) are generating the most revenue for italian restaurant 
select l.city,round(sum(o.total_amount),2) as total_revenue from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id join locations_filtered l on o.delivery_location_id=l.location_id 
where r.cuisines ilike 'Italian' group by l.city order by 2 desc 

city,total_revenue
South Steven,4077.09
North Jeff,3811.75
Port Lisafort,3759.01
West Jessica,3659.54
North Sarah,3655.7
Harperville,3601.73
New Kathy,3598.22
Roberthaven,3587.16
Allisonmouth,3499.69
Weeksview,3393.3


In [0]:
-- 6.Which cities (area_name) are generating the most revenue for italian restaurant 
-- select l.city,round(sum(o.total_amount),2) as total_revenue from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id join locations_filtered l on o.delivery_location_id=l.location_id 
-- where r.cuisines ilike '%italian' group by l.city order by total_revenue desc 

In [0]:
-- 7. what is the tip-to-total ratio by cusine type
select  r.cuisines,round(sum(o.tip)/sum(o.total_amount),2) as ratio from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id 
group by r.cuisines 

cuisines,ratio
Italian,0.03
Indian,0.03
Thai,0.03
Chinese,0.03
Mexican,0.03
American,0.03


In [0]:
create or replace temp view orders_exploded as 
select agent_id,customer_id,delivery_location_id,explode(items_ordered.item_id) as item_id ,explode(items_ordered.item_name) as item_name,explode(items_ordered.price) as price,explode(items_ordered.quantity) as quantity,order_id,order_status,payment_method,restaurant_id,tip,total_amount
from orders_bronze

In [0]:
-- 8.Which menu items are most frequently ordered and from which restaurants 
select o.item_name,m.restaurant_id,count(o.order_id) as cnt from orders_exploded o join menu_items_filtered m on o.item_id=m.item_id group by o.item_name,m.restaurant_id
order by cnt desc 

item_name,restaurant_id,cnt
Seem Garlic Bread,R171,1390
Commercial Pizza,R171,1384
Huge Brownie,R171,1359
Ok Spring Roll,R146,1269
Trade Pizza,R146,1248
Magazine Wrap,R105,1226
She Tea,R146,1226
Statement Coffee,R079,1191
Commercial Brownie,R035,1174
Recognize Pasta,R105,1174


In [0]:
-- 9.which agents have delivered the most diverse cuisine
select a.agent_id,count(distinct r.cuisines) as unq_cnt from orders_bronze o join delivery_agents_filtered a on o.agent_id=a.agent_id join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id  group by a.agent_id,r.restaurant_id order by unq_cnt desc 

agent_id,unq_cnt
A390,3
A148,3
A315,3
A296,3
A060,3
A202,3
A274,3
A203,3
A283,3
A272,3


In [0]:
-- 10. What is the avg customer lifetime value by signup year 
with cte as (select c.customer_id,extract(year from c.signup_date) as signup_year ,round(sum(o.total_amount)) as total from orders_bronze o join customers_filtered c on o.customer_id=c.customer_id group by c.customer_id,extract(year from c.signup_date))
select signup_year, round(avg(total),2) as avg_total from cte group by signup_year order by signup_year;

signup_year,avg_total
1954,23.0
1955,173.0
1956,140.0
1957,155.57
1958,135.5
1959,141.14
1960,173.07
1961,222.21
1962,192.0
1963,137.3


In [0]:
-- 11.Which restaurants have the highest order frequency per location 
select r.restaurant_id,l.location_id,count(*) as order_cnt from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id join locations_filtered l on o.delivery_location_id=l.location_id group by r.restaurant_id,l.location_id order by order_cnt desc

restaurant_id,location_id,order_cnt
R042,L051,6
R196,L090,6
R053,L065,6
R159,L012,6
R148,L029,6
R140,L083,5
R038,L015,5
R043,L041,5
R092,L008,5
R014,L072,5


In [0]:
-- 12.Which locations are prone to order cancellations
select l.area, count(*) as order_cnt from orders_bronze o join locations_filtered l on o.delivery_location_id=l.location_id where order_status='cancelled' group by l.area order by l.area, order_cnt 

area,order_cnt
Adams Fort,37
Aguilar Pass,45
Alex Curve,44
Alex Mission,43
Andrews Heights,45
Angela Lodge,43
Anthony Forge,41
Arnold Fort,50
Ashley Extensions,53
Boyle Dam,35


In [0]:
-- 13.Find top 10 customers who have ordered from the wildest variety of restaurants 
select o.customer_id,count(distinct r.restaurant_id) as variety_res from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id group by o.customer_id order by variety_res desc limit 10

customer_id,variety_res
C7381,7
C5365,7
C6470,7
C7526,7
C1423,6
C0362,6
C6971,6
C2116,6
C4165,6
C1449,6


In [0]:
-- 14. Identify restaurants with frequent repeat customers
select r.restaurant_id,o.customer_id,count(*) as order_cnt from orders_bronze o join restaurants_filtered r on  o.restaurant_id=r.restaurant_id 
group by 1,2 
having order_cnt >1 

restaurant_id,customer_id,order_cnt
R148,C7659,2
R038,C9577,2
R055,C9140,2
R066,C5987,2
R150,C0119,2
R019,C1838,2
R053,C2179,2
R019,C4924,2
R181,C3734,2
R029,C2966,2


In [0]:
-- 15. Which Payment methods are most preferred for high -value orders across cuisines
select r.cuisines,o.payment_method,count(*) as order_cnt from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id where o.total_amount >40 
group by r.cuisines,o.payment_method  order by order_cnt desc 

cuisines,payment_method,order_cnt
Mexican,cash,833
American,cash,812
American,wallet,803
American,card,799
American,UPI,794
Mexican,card,775
Mexican,UPI,770
Mexican,wallet,762
Thai,wallet,725
Thai,card,691


In [0]:
drop schema kishoredb.streamdb cascade;
create schema kishoredb.streamdb;